# Ingesta de Markdown en colecciones de Qdrant

Este cuaderno toma una carpeta de archivos Markdown y los ingiere de verdad con
`hermes-ingest-md` en **tantas colecciones de Qdrant como declare la
configuración**. Cada colección tiene sus propios documentos, su propio
troceado y su propio embedder. Al final las colecciones quedan listas para
evaluarlas con `rag_evaluation.ipynb`.

Cómo se usa:

1. Ejecuta las celdas en orden. Cada celda muestra su resultado intermedio.
2. Para otra ingesta cambia solo el diccionario `CONFIG` (sección 2) y vuelve a
   ejecutar desde ahí.
3. Con `HERMES_INGEST_OFFLINE=1` todo corre sin Ollama ni Qdrant (embedder
   hash, Qdrant en memoria, corpus de ejemplo): sirve para ver que el cuaderno
   funciona, no para medir calidad.

El cuaderno no trocea ni calcula embeddings por su cuenta: usa las funciones
del paquete (`hermes_report.corpus`, `.ingest`, `.store`). Solo crea o
escribe colecciones cuyo nombre empieza por `hermes_report_`; el resto de
colecciones de Qdrant solo se lee.

Da información mientras trabaja: una línea por documento durante la ingesta,
tablas de estadísticas por colección y por documento, y una foto de Qdrant
**antes** y **después** con las diferencias.

## 1. Preparación

Importaciones, modo (con o sin conexión) y carpeta de salida. La carpeta de
salida es `HERMES_INGEST_OUT` o, si no existe, `notebooks/out/ingest/`. Ahí
quedan la caché de ingesta (`ingest-cache/`) y el manifiesto
(`collections.json`).

In [ ]:
# Section 1: imports, mode and output folder.
import json
import os
import statistics
import time
from dataclasses import asdict, fields
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from hermes_enrich.tokens import default_token_counter
from IPython.display import display

from hermes_report.config import EmbeddingConfig, IngestConfig
from hermes_report.corpus import Corpus, CorpusError, load_corpus
from hermes_report.ingest import ingest_corpus
from hermes_report.providers import check_ollama, make_embedder
from hermes_report.store import PREFIX, ensure_collection, make_client

# Offline mode replaces every service by a local stand-in (no Ollama, no Qdrant).
OFFLINE = os.environ.get("HERMES_INGEST_OFFLINE") == "1"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)


def find_root() -> Path:
  """The repository folder: the nearest parent of the cwd that holds the demo."""
  here = Path.cwd().resolve()
  for folder in (here, *here.parents):
    if (folder / "examples" / "demo" / "corpus").is_dir():
      return folder
  raise RuntimeError("Abre el cuaderno dentro del repositorio hermes-report-generator.")


ROOT = find_root()
OUT_DIR = Path(
  os.environ.get("HERMES_INGEST_OUT") or ROOT / "notebooks" / "out" / "ingest"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

display(
  {"modo": "sin conexión" if OFFLINE else "con Ollama y Qdrant", "salida": str(OUT_DIR)}
)

## 2. Configuración de la ingesta

Todo lo que cambia entre ingestas está en `CONFIG`:

| Clave | Qué es |
|---|---|
| `corpus_dir` | carpeta con los Markdown (relativa a la raíz del repositorio, o absoluta) |
| `qdrant_url` | URL de Qdrant (se ignora sin conexión) |
| `force` | `True` reconstruye todas las colecciones aunque no hayan cambiado |
| `defaults.ingest` | troceado común: `max_chunk_size`, `hard_boundary_levels`, `strip_frontmatter` |
| `defaults.embedding` | embedder común: `provider` (`ollama` o `hash`) y `model` |
| `collections` | una entrada por colección; el nombre debe empezar por `hermes_report_` |
| `collections.<nombre>.documents` | nombres exactos de los documentos (ruta relativa a `corpus_dir`, sin patrones) |
| `collections.<nombre>.ingest` / `.embedding` | opcionales: se mezclan sobre los valores por defecto |

Una colección se **reutiliza** si ya contiene exactamente los mismos puntos y se
**reconstruye** en caso contrario (o siempre con `force`). La caché de ingesta
evita volver a calcular embeddings de documentos que no han cambiado.

Variables de entorno opcionales (para ejecutar sin abrir el cuaderno):

- `HERMES_INGEST_CONFIG`: ruta de un archivo JSON con la misma forma que
  `CONFIG`; lo sustituye por completo.
- `HERMES_INGEST_OUT`: carpeta de salida.
- `HERMES_INGEST_OFFLINE=1`: modo sin conexión. Fuerza `qdrant_url` a
  `":memory:"`, todos los embedders a `hash` y `corpus_dir` a
  `examples/demo/corpus` (salvo que el JSON lo indique). Sin JSON, usa dos
  colecciones de ejemplo sobre ese corpus.

In [ ]:
# Section 2: the one place to edit between ingestions.
CONFIG: dict[str, Any] = {
  "corpus_dir": "notebooks/data",
  "qdrant_url": "http://localhost:6333",
  "force": False,  # rebuild every collection even if unchanged
  "defaults": {
    "ingest": {
      "max_chunk_size": 1500,
      "hard_boundary_levels": [1, 2],
      "strip_frontmatter": True,
    },
    "embedding": {"provider": "ollama", "model": "nomic-embed-text"},
  },
  "collections": {
    "hermes_report_golden_agents": {
      "documents": ["agentes-ia-arquitecturas.md"],
      "ingest": {"max_chunk_size": 800},  # optional, merged over defaults
    },
    "hermes_report_golden_python": {
      "documents": [
        "fastapi-readme.md",
        "flask-readme.md",
        "pydantic-readme.md",
        "typer-readme.md",
      ],
    },
  },
}

# Optional: a JSON file replaces CONFIG (so a run needs no notebook edit).
CONFIG_FILE = os.environ.get("HERMES_INGEST_CONFIG")
if CONFIG_FILE:
  config_path = Path(CONFIG_FILE)
  if not config_path.is_absolute() and not config_path.exists():
    config_path = ROOT / config_path
  try:
    CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
  except (OSError, ValueError) as exc:
    raise RuntimeError(
      f"No se puede leer HERMES_INGEST_CONFIG ({config_path}): {exc}"
    ) from exc
  if not isinstance(CONFIG, dict):
    raise RuntimeError(
      f"{config_path}: el JSON debe ser un objeto con las claves de CONFIG"
    )

# Offline: Qdrant in memory, hash embedders and the demo corpus.
HASH_EMBEDDING = {"provider": "hash", "model": "hash-256"}
if OFFLINE:
  CONFIG["qdrant_url"] = ":memory:"
  if not (CONFIG_FILE and "corpus_dir" in CONFIG):
    CONFIG["corpus_dir"] = "examples/demo/corpus"
  if "defaults" not in CONFIG or isinstance(CONFIG["defaults"], dict):
    CONFIG["defaults"] = {**CONFIG.get("defaults", {}), "embedding": HASH_EMBEDDING}
  if not CONFIG_FILE:
    CONFIG["collections"] = {
      PREFIX + "demo_manuales": {
        "documents": ["manuales/brujula.md", "manuales/faro.md"],
        "ingest": {"max_chunk_size": 300},
      },
      PREFIX + "demo_todo": {
        "documents": [
          "manuales/faro.md",
          "politicas/teletrabajo.md",
          "politicas/vacaciones.md",
        ],
        "ingest": {"max_chunk_size": 600},
      },
    }
  if isinstance(CONFIG.get("collections"), dict):
    for spec in CONFIG["collections"].values():
      if isinstance(spec, dict) and "embedding" in spec:
        spec["embedding"] = HASH_EMBEDDING

display(CONFIG)

## 3. Validación

Antes de tocar Qdrant u Ollama se comprueba la configuración. Si algo está mal
la celda falla con un mensaje que lista **todos** los problemas:

- hay al menos una colección y todas empiezan por `hermes_report_`;
- cada colección tiene una lista `documents` no vacía y sin repetidos;
- cada documento existe en la carpeta (nombre exacto, relativo a `corpus_dir`);
  el error muestra los desconocidos y los disponibles;
- las claves de `ingest` y `embedding` son las que existen
  (`max_chunk_size`, `hard_boundary_levels`, `strip_frontmatter` y `provider`,
  `model`) y sus valores son razonables.

Si todo es válido, muestra el plan: una fila por colección y los documentos de
la carpeta que ninguna colección usa.

In [ ]:
# Section 3: validate CONFIG before touching Qdrant or Ollama.
problems: list[str] = []
INGEST_KEYS = {f.name for f in fields(IngestConfig)}
EMBEDDING_KEYS = {f.name for f in fields(EmbeddingConfig)}
TOP_KEYS = {"corpus_dir", "qdrant_url", "force", "defaults", "collections"}


def section(where: str, value: Any, allowed: set[str]) -> dict[str, Any]:
  """A configuration mapping with only the ``allowed`` keys (else a problem)."""
  if value is None:
    return {}
  if not isinstance(value, dict):
    problems.append(f"{where}: debe ser un objeto, no {type(value).__name__}")
    return {}
  unknown = sorted(set(value) - allowed)
  if unknown:
    problems.append(
      f"{where}: claves desconocidas {unknown}; las válidas son {sorted(allowed)}"
    )
  return {k: v for k, v in value.items() if k in allowed}


def build_ingest(where: str, values: dict[str, Any]) -> IngestConfig | None:
  size = values.get("max_chunk_size", IngestConfig.max_chunk_size)
  levels = values.get("hard_boundary_levels", IngestConfig.hard_boundary_levels)
  strip = values.get("strip_frontmatter", IngestConfig.strip_frontmatter)
  start = len(problems)
  if isinstance(size, bool) or not isinstance(size, int) or size <= 0:
    problems.append(f"{where}: max_chunk_size debe ser un entero positivo, no {size!r}")
  if not isinstance(levels, list | tuple) or not all(
    isinstance(n, int) and not isinstance(n, bool) and 1 <= n <= 6 for n in levels
  ):
    problems.append(
      f"{where}: hard_boundary_levels debe ser una lista de niveles 1 a 6"
    )
  if not isinstance(strip, bool):
    problems.append(f"{where}: strip_frontmatter debe ser true o false")
  if len(problems) > start:
    return None
  return IngestConfig(size, tuple(levels), strip)


def build_embedding(where: str, values: dict[str, Any]) -> EmbeddingConfig | None:
  provider = values.get("provider", EmbeddingConfig.provider)
  model = values.get("model", EmbeddingConfig.model)
  if provider not in ("ollama", "hash"):
    problems.append(f"{where}: provider debe ser 'ollama' o 'hash', no {provider!r}")
    return None
  if not isinstance(model, str) or not model.strip():
    problems.append(f"{where}: model debe ser un texto no vacío")
    return None
  return EmbeddingConfig(provider, model)


if not isinstance(CONFIG, dict):
  raise ValueError("Configuración no válida: CONFIG debe ser un objeto")
unknown_top = sorted(set(CONFIG) - TOP_KEYS)
if unknown_top:
  problems.append(
    f"claves desconocidas {unknown_top}; las válidas son {sorted(TOP_KEYS)}"
  )
if not isinstance(CONFIG.get("qdrant_url"), str) or not CONFIG["qdrant_url"].strip():
  problems.append("qdrant_url debe ser un texto no vacío")
if not isinstance(CONFIG.get("force", False), bool):
  problems.append("force debe ser true o false")

# The folder, loaded exactly as the pipeline loads it.
corpus: Corpus | None = None
corpus_dir = CONFIG.get("corpus_dir")
if not isinstance(corpus_dir, str) or not corpus_dir.strip():
  problems.append("corpus_dir debe ser la ruta de una carpeta")
else:
  try:
    corpus = load_corpus(ROOT / corpus_dir)
  except CorpusError as exc:
    problems.append(str(exc))
available = list(corpus.sources) if corpus else []

defaults = section("defaults", CONFIG.get("defaults"), {"ingest", "embedding"})
default_ingest = section("defaults.ingest", defaults.get("ingest"), INGEST_KEYS)
default_embedding = section(
  "defaults.embedding", defaults.get("embedding"), EMBEDDING_KEYS
)

collections = CONFIG.get("collections")
if not isinstance(collections, dict) or not collections:
  problems.append("collections debe tener al menos una colección")
  collections = {}

# PLAN: per collection, the documents and the resolved chunker and embedder.
PLAN: dict[str, dict[str, Any]] = {}
for name, spec in collections.items():
  if not isinstance(name, str) or not name.startswith(PREFIX):
    problems.append(f"{name!r}: el nombre de la colección debe empezar por {PREFIX!r}")
  fields_ok = section(f"{name}", spec, {"documents", "ingest", "embedding"})
  documents = fields_ok.get("documents")
  if not isinstance(documents, list) or not documents:
    problems.append(f"{name}: documents debe ser una lista no vacía")
    documents = []
  elif not all(isinstance(d, str) and d for d in documents):
    problems.append(f"{name}: documents solo puede contener nombres de archivo")
    documents = []
  repeated = sorted({d for d in documents if documents.count(d) > 1})
  if repeated:
    problems.append(f"{name}: documentos repetidos {repeated}")
  if corpus is not None:
    unknown_docs = [d for d in documents if d not in available]
    if unknown_docs:
      problems.append(
        f"{name}: documentos desconocidos {unknown_docs}; "
        f"los disponibles en {corpus_dir} son {available}"
      )
  ingest_values = {
    **default_ingest,
    **section(f"{name}.ingest", fields_ok.get("ingest"), INGEST_KEYS),
  }
  embedding_values = {
    **default_embedding,
    **section(f"{name}.embedding", fields_ok.get("embedding"), EMBEDDING_KEYS),
  }
  ingest_config = build_ingest(f"{name}.ingest", ingest_values)
  embedding_config = build_embedding(f"{name}.embedding", embedding_values)
  PLAN[name] = {
    "documents": documents,
    "ingest": ingest_config,
    "embedding": embedding_config,
  }

if problems:
  raise ValueError(
    "Configuración no válida, no se ha tocado Qdrant ni Ollama:\n"
    + "\n".join(f"  - {p}" for p in problems)
  )
assert corpus is not None

CONFIG.setdefault("force", False)
CHARS = {doc.source: doc.chars for doc in corpus.documents}


def chunk_settings(config: IngestConfig) -> str:
  frontmatter = "sí" if config.strip_frontmatter else "no"
  levels = list(config.hard_boundary_levels)
  return f"máx {config.max_chunk_size}, cortes {levels}, frontmatter {frontmatter}"


def embedder_label(config: EmbeddingConfig) -> str:
  return f"{config.provider}:{config.model}"


plan_table = pd.DataFrame(
  [
    {
      "colección": name,
      "documentos": len(plan["documents"]),
      "troceado": chunk_settings(plan["ingest"]),
      "embedder": embedder_label(plan["embedding"]),
    }
    for name, plan in PLAN.items()
  ]
)
display(plan_table)

used = {d for plan in PLAN.values() for d in plan["documents"]}
unused_table = pd.DataFrame(
  [
    {"documento": doc.source, "caracteres": doc.chars, "sha256": doc.sha256[:12]}
    for doc in corpus.documents
    if doc.source not in used
  ]
)
if unused_table.empty:
  print("Todos los documentos de la carpeta se usan en alguna colección.")
else:
  print("Documentos de la carpeta que ninguna colección usa:")
  display(unused_table)

## 4. Servicios

Con conexión se comprueba que Ollama responde y tiene cada modelo de
embeddings distinto que pide la configuración (así un modelo que falta falla
aquí y no a mitad de la ingesta). Se crea **un embedder por cada configuración
de embeddings distinta** y el cliente de Qdrant. Sin conexión solo se crean el
embedder hash y el Qdrant en memoria.

In [ ]:
# Section 4: Ollama check, one embedder per distinct embedding config, Qdrant client.
distinct_embeddings = {
  (plan["embedding"].provider, plan["embedding"].model): plan["embedding"]
  for plan in PLAN.values()
}
if not OFFLINE:
  ollama_models = sorted(
    model for provider, model in distinct_embeddings if provider == "ollama"
  )
  if ollama_models:
    check_ollama(ollama_models)

embedders = {key: make_embedder(config) for key, config in distinct_embeddings.items()}
client = make_client(CONFIG["qdrant_url"])
counter = default_token_counter()

display(
  {
    "qdrant": CONFIG["qdrant_url"],
    "embedders": sorted(f"{provider}:{model}" for provider, model in embedders),
  }
)

## 5. Antes: lo que hay en Qdrant

Foto de **todas** las colecciones que Qdrant tiene antes de ingerir (solo
lectura): puntos, tamaño del vector y distancia. La columna `configurada` dice
si la colección está en `CONFIG` y `acción` qué hará esta ejecución con ella:
comprobar si se reutiliza, reconstruirla (`force`) o nada. Las colecciones que
no empiezan por `hermes_report_` aparecen en la lista pero **nunca se tocan**.
Las colecciones configuradas que aún no existen se anuncian debajo de la tabla.

In [ ]:
# Section 5: read-only snapshot of every collection Qdrant holds (and the plan).
def snapshot(action_of: Any) -> list[dict[str, Any]]:
  """One row per collection Qdrant holds; only reads."""
  rows = []
  try:
    for name in sorted(c.name for c in client.get_collections().collections):
      vectors = client.get_collection(name).config.params.vectors
      if isinstance(vectors, dict):
        size: Any = "con nombre"
        distance = "-"
      else:
        size = vectors.size
        distance = str(getattr(vectors.distance, "value", vectors.distance))
      rows.append(
        {
          "colección": name,
          "puntos": client.count(name, exact=True).count,
          "tamaño vector": size,
          "distancia": distance,
          "configurada": "sí" if name in PLAN else "no",
          "acción": action_of(name),
        }
      )
  except Exception as exc:
    raise RuntimeError(f"Qdrant no responde en {CONFIG['qdrant_url']}: {exc}") from exc
  return rows


def planned_action(name: str) -> str:
  if name in PLAN:
    return "reconstruir (force)" if CONFIG["force"] else "comprobar si se reutiliza"
  if name.startswith(PREFIX):
    return "sin tocar (no configurada)"
  return "nunca se toca"


SNAPSHOT_COLUMNS = [
  "colección",
  "puntos",
  "tamaño vector",
  "distancia",
  "configurada",
  "acción",
]
BEFORE = snapshot(planned_action)
display(pd.DataFrame(BEFORE, columns=SNAPSHOT_COLUMNS))
new_names = [name for name in PLAN if name not in {row["colección"] for row in BEFORE}]
if new_names:
  print("Colecciones que se crearán:", ", ".join(new_names))

## 6. Ingesta

Para cada colección se ingieren **solo sus documentos** con su troceado y su
embedder, y después se crea la colección en Qdrant (o se reutiliza si ya tiene
exactamente esos puntos). Se imprime una cabecera por colección
(`[2/3] nombre: 4 documentos`), una línea por documento (chunks, si salió de la
caché o se ingirió, y segundos) y una línea final con el resultado: `creada`,
`recreada` o `reutilizada`, los puntos y los segundos.

La caché (`ingest-cache/` en la carpeta de salida) guarda los puntos por
documento: un documento que no ha cambiado no vuelve a calcular embeddings, ni
siquiera cuando `force` reconstruye la colección. Si una colección falla se
muestra su error y las demás continúan.

In [ ]:
# Section 6: ingest each collection with live progress, then create or reuse it.
class ProgressPrinter:
  """Prints one line per document, with the seconds since the previous one."""

  def __init__(self) -> None:
    self.last = time.perf_counter()

  def reset(self) -> None:
    self.last = time.perf_counter()

  def __call__(self, message: str) -> None:
    now = time.perf_counter()
    print(f"  - {message}, {now - self.last:.2f} s")
    self.last = now


RESULTS: dict[str, dict[str, Any]] = {}
printer = ProgressPrinter()
names = list(PLAN)
run_started = time.perf_counter()
for number, name in enumerate(names, start=1):
  plan = PLAN[name]
  wanted = set(plan["documents"])
  subset = Corpus(
    dir=corpus.dir,
    glob=corpus.glob,
    documents=tuple(d for d in corpus.documents if d.source in wanted),
  )
  noun = "documento" if len(subset.documents) == 1 else "documentos"
  print(f"[{number}/{len(names)}] {name}: {len(subset.documents)} {noun}")
  started = time.perf_counter()
  printer.reset()
  try:
    ingestion = ingest_corpus(
      subset,
      plan["ingest"],
      plan["embedding"],
      embedders[(plan["embedding"].provider, plan["embedding"].model)],
      OUT_DIR / "ingest-cache",
      progress=printer,
    )
    existed = client.collection_exists(name)
    info = ensure_collection(
      client, name, ingestion.points, CONFIG["qdrant_url"], force=CONFIG["force"]
    )
  except Exception as exc:
    seconds = time.perf_counter() - started
    RESULTS[name] = {
      "outcome": "fallida",
      "error": f"{type(exc).__name__}: {exc}",
      "seconds": seconds,
      "ingestion": None,
      "info": None,
    }
    print(f"  ERROR ({seconds:.2f} s): {exc}")
    continue
  seconds = time.perf_counter() - started
  outcome = (
    "reutilizada" if not info.reloaded else ("recreada" if existed else "creada")
  )
  RESULTS[name] = {
    "outcome": outcome,
    "error": None,
    "seconds": seconds,
    "ingestion": ingestion,
    "info": info,
  }
  print(f"  -> {outcome}: {info.points} puntos, {seconds:.2f} s")
TOTAL_SECONDS = time.perf_counter() - run_started
print(f"Total: {TOTAL_SECONDS:.2f} s")

## 7. Estadísticas

Tres tablas de lo que se ha ingerido:

1. **Por colección**: documentos, chunks, chunks por documento
   (mín / media / máx), tamaño de los chunks en caracteres y en tokens
   (mín / media / mediana / máx, con el contador de tokens por defecto de
   `hermes_enrich`), tamaño del vector, embedder, troceado, documentos que
   salieron de la caché frente a los ingeridos, segundos y resultado.
2. **Por documento y colección**: chunks, caracteres, tamaño medio de chunk y
   profundidad de los encabezados (`heading_path`) cuando el payload la trae.
3. **Documentos sin usar**: los de la carpeta que ninguna colección ingirió.

In [ ]:
# Section 7a: statistics per collection.
def spread(values: list[int], median: bool = True) -> str:
  """min / mean / [median /] max of ``values`` as one short text."""
  if not values:
    return "-"
  parts = [min(values), round(statistics.fmean(values))]
  if median:
    parts.append(round(statistics.median(values)))
  parts.append(max(values))
  return " / ".join(str(p) for p in parts)


def chunk_texts(ingestion: Any) -> list[str]:
  return [str(point.payload.get("text", "")) for point in ingestion.points]


collection_rows = []
for name, result in RESULTS.items():
  plan = PLAN[name]
  row: dict[str, Any] = {
    "colección": name,
    "documentos": len(plan["documents"]),
    "chunks": None,
    "chunks/doc (mín/media/máx)": "-",
    "caracteres (mín/media/mediana/máx)": "-",
    "tokens (mín/media/mediana/máx)": "-",
    "vector": None,
    "embedder": embedder_label(plan["embedding"]),
    "troceado": chunk_settings(plan["ingest"]),
    "caché/ingeridos": "-",
    "segundos": round(result["seconds"], 2),
    "resultado": result["outcome"],
  }
  ingestion = result["ingestion"]
  if ingestion is not None:
    texts = chunk_texts(ingestion)
    cached = sum(1 for doc in ingestion.documents if doc.cached)
    row.update(
      {
        "chunks": len(texts),
        "chunks/doc (mín/media/máx)": spread(
          [len(doc.points) for doc in ingestion.documents], median=False
        ),
        "caracteres (mín/media/mediana/máx)": spread([len(t) for t in texts]),
        "tokens (mín/media/mediana/máx)": spread([counter.count(t) for t in texts]),
        "vector": result["info"].vector_size,
        "caché/ingeridos": f"{cached}/{len(ingestion.documents) - cached}",
      }
    )
  collection_rows.append(row)
collection_stats = pd.DataFrame(collection_rows)
display(collection_stats)

In [ ]:
# Section 7b: statistics per document and collection.
document_rows = []
for name, result in RESULTS.items():
  ingestion = result["ingestion"]
  if ingestion is None:
    continue
  for doc in ingestion.documents:
    sizes = [len(str(p.payload.get("text", ""))) for p in doc.points]
    depths = [
      len(p.payload["heading_path"])
      for p in doc.points
      if isinstance(p.payload.get("heading_path"), list)
    ]
    document_rows.append(
      {
        "colección": name,
        "documento": doc.source,
        "chunks": len(doc.points),
        "caracteres": CHARS[doc.source],
        "chunk medio": round(statistics.fmean(sizes)) if sizes else 0,
        "profundidad encabezados": (
          "-" if not depths else f"{min(depths)}-{max(depths)}"
        ),
        "origen": "caché" if doc.cached else "ingerido",
        "segundos": round(doc.seconds, 2),
      }
    )
document_stats = pd.DataFrame(document_rows)
display(document_stats)

In [ ]:
# Section 7c: documents of the folder that no collection ingested.
ingested_sources = {
  doc.source
  for result in RESULTS.values()
  if result["ingestion"] is not None
  for doc in result["ingestion"].documents
}
unused_documents = pd.DataFrame(
  [
    {"documento": doc.source, "caracteres": doc.chars}
    for doc in corpus.documents
    if doc.source not in ingested_sources
  ]
)
if unused_documents.empty:
  print("Todos los documentos de la carpeta se han ingerido en alguna colección.")
else:
  print("Documentos de la carpeta que ninguna colección ha usado:")
  display(unused_documents)

## 8. Después: Qdrant y diferencias

La misma foto de Qdrant tras ingerir (la columna `acción` ahora es el
resultado) y una tabla de diferencias con una fila por colección:

| Estado | Significado |
|---|---|
| `nueva` | no existía y se ha creado |
| `reconstruida` | existía con otros puntos (o con `force`); se muestran los puntos antes y después |
| `reutilizada sin cambios` | ya tenía exactamente esos puntos |
| `fallida` | la ingesta o la creación dieron error |
| `sin tocar` | colección de Qdrant que esta ejecución no usa; sus puntos no deben cambiar |

Debajo, los totales: colecciones y puntos antes y después, documentos
ingeridos, chunks calculados frente a leídos de la caché y segundos.

In [ ]:
# Section 8: snapshot after ingesting, per-collection diff and totals.
def final_action(name: str) -> str:
  return RESULTS[name]["outcome"] if name in RESULTS else planned_action(name)


AFTER = snapshot(final_action)
display(pd.DataFrame(AFTER, columns=SNAPSHOT_COLUMNS))

STATE = {
  "creada": "nueva",
  "recreada": "reconstruida",
  "reutilizada": "reutilizada sin cambios",
  "fallida": "fallida",
}
points_before = {row["colección"]: row["puntos"] for row in BEFORE}
points_after = {row["colección"]: row["puntos"] for row in AFTER}
diff_rows = []
for name in sorted(set(points_before) | set(points_after) | set(PLAN)):
  before_points = points_before.get(name, 0)
  after_points = points_after.get(name, 0)
  if name in RESULTS:
    state = STATE[RESULTS[name]["outcome"]]
    detail = RESULTS[name]["error"] or f"{after_points} puntos"
    if state == "reconstruida":
      detail = f"{before_points} → {after_points} puntos"
  else:
    state = "sin tocar"
    detail = f"{after_points} puntos"
    if before_points != after_points:
      detail = f"¡han cambiado! {before_points} → {after_points} puntos"
  diff_rows.append(
    {
      "colección": name,
      "estado": state,
      "puntos antes": before_points,
      "puntos después": after_points,
      "detalle": detail,
    }
  )
diff = pd.DataFrame(diff_rows)
display(diff)

done = [r["ingestion"] for r in RESULTS.values() if r["ingestion"] is not None]
embedded_chunks = sum(len(d.points) for i in done for d in i.documents if not d.cached)
cached_chunks = sum(len(d.points) for i in done for d in i.documents if d.cached)
totals = pd.DataFrame(
  [
    ("colecciones antes → después", f"{len(BEFORE)} → {len(AFTER)}"),
    (
      "puntos antes → después",
      f"{sum(points_before.values())} → {sum(points_after.values())}",
    ),
    ("documentos ingeridos", sum(len(i.documents) for i in done)),
    ("chunks calculados (embeddings)", embedded_chunks),
    ("chunks leídos de la caché", cached_chunks),
    ("segundos totales", round(TOTAL_SECONDS, 2)),
  ],
  columns=["total", "valor"],
)
display(totals)

## 9. Manifiesto

Se guarda `collections.json` en la carpeta de salida con, por colección: sus
documentos con su `sha256`, el troceado y los embeddings resueltos, los puntos,
el resultado, la URL de Qdrant y la hora; además las fotos de antes y después.
Al final se imprime la lista de colecciones listas para pegar en la clave
`collections` de `CONFIG` en `rag_evaluation.ipynb`.

In [ ]:
# Section 9: the manifest and the collections ready for rag_evaluation.ipynb.
SHA = {doc.source: doc.sha256 for doc in corpus.documents}
stamp = datetime.now().astimezone().isoformat(timespec="seconds")
manifest: dict[str, Any] = {
  "timestamp": stamp,
  "qdrant_url": CONFIG["qdrant_url"],
  "corpus_dir": str(corpus.dir),
  "force": CONFIG["force"],
  "collections": {},
  "before": BEFORE,
  "after": AFTER,
}
for name, result in RESULTS.items():
  plan = PLAN[name]
  ingestion = result["ingestion"]
  by_source = {doc.source: doc for doc in ingestion.documents} if ingestion else {}
  embedding_info = asdict(plan["embedding"])
  if ingestion is not None:
    embedding_info["dimension"] = ingestion.embedding["dimension"]
  manifest["collections"][name] = {
    "documents": [
      {
        "source": source,
        "sha256": SHA[source],
        "chunks": len(by_source[source].points) if source in by_source else None,
      }
      for source in sorted(plan["documents"])
    ],
    "ingest": asdict(plan["ingest"]),
    "embedding": embedding_info,
    "points": result["info"].points if result["info"] else None,
    "outcome": result["outcome"],
    "error": result["error"],
    "qdrant_url": CONFIG["qdrant_url"],
    "timestamp": stamp,
  }
manifest_path = OUT_DIR / "collections.json"
manifest_path.write_text(
  json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(f"Manifiesto guardado en {manifest_path}")

READY = [name for name, result in RESULTS.items() if result["outcome"] != "fallida"]
print("Para rag_evaluation.ipynb, en CONFIG['collections']:")
print(json.dumps(READY, ensure_ascii=False))